# FLEMMS 2024 Exploratory Data Analysis

Scope: general EDA of the merged PSA FLEMMS 2024 microdata (ages 10 to 64), followed by PCA diagnostics.
All rates are survey-weighted with `RESP_RFACT_F2` unless labelled unweighted. Target is `y` = 1 if functionally illiterate (`FLITERATE == 2`), 0 if literate.

Sections
1. Setup and load
2. Data audit
3. Missingness and skip logic
4. Target distribution (weighted vs unweighted)
5. Univariate profile
6. Bivariate: features vs target
7. Association ranking (Cramer's V)
8. Geography
9. Feature-feature association
10. PCA diagnostics

RAM note: sections 1 to 9 run on the raw merged frame (no one-hot). Only section 10 one-hot encodes, with the same memory-safe logic as `PCA_FLEMMS_streamlined.ipynb`.

## 1. Setup and load

In [ ]:
import gc
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RNG = np.random.RandomState(42)

FIG_DIR = 'eda_figures'
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=200, bbox_inches='tight')
    plt.show()

RTF1_FILE = r'data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - RTF1.CSV'
RTF2_FILE = r'data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - RTF2.CSV'
RTF3_FILE = r'data/PHL-PSA-FLEMMS-2024-V2-PUF/FLEMMS PUF 2024 Volume2 - RTF3.CSV'
MEMBER_FILE = r'data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - MEMBER.CSV'
V1_DICT_FILE = r'data/PHL-PSA-FLEMMS-2024-V1-PUF/flemms_2024_v1_metadata(dictionary).xlsx'
V2_DICT_FILE = r'data/PHL-PSA-FLEMMS-2024-V2-PUF/flemms_2024_v2_metadata(dictionary).xlsx'
PHASE1_FILE = 'phase1_selected_features.csv'
PHASE1B_FILE = 'phase1b_selected_features.csv'

W = 'RESP_RFACT_F2'

In [ ]:
def load_and_merge_flemms():
    # Same merge logic as PCA_FLEMMS_streamlined.ipynb (Phase 1).
    # skipinitialspace: PSA pads codes with spaces (' 1') and blanks (' '), so those columns load as text.
    # Stripping makes blanks NaN and codes numeric, so FLITERATE maps and missingness counts are correct.
    df_member = pd.read_csv(MEMBER_FILE, skipinitialspace=True)
    df_rtf1 = pd.read_csv(RTF1_FILE, skipinitialspace=True)
    df_rtf2 = pd.read_csv(RTF2_FILE, skipinitialspace=True)
    df_rtf3 = pd.read_csv(RTF3_FILE, skipinitialspace=True)

    overlap_cols = ['REG', 'PRV', 'MUN', 'BGY', 'URBANITY', 'REG2']
    def remove_overlaps(df, merge_keys):
        return df.drop(columns=[c for c in overlap_cols if c in df.columns and c not in merge_keys])

    df_rtf1 = remove_overlaps(df_rtf1, ['HHID'])
    if 'LNO_F2' in df_rtf2.columns:
        df_rtf2 = df_rtf2.rename(columns={'LNO_F2': 'LNO'})
    df_rtf2 = remove_overlaps(df_rtf2, ['HHID', 'LNO'])
    if 'LNO_F3' in df_rtf3.columns:
        df_rtf3 = df_rtf3.rename(columns={'LNO_F3': 'LNO'})
    df_rtf3 = remove_overlaps(df_rtf3, ['HHID', 'LNO'])

    n_member = len(df_member)
    df = df_member.merge(df_rtf1, on='HHID', how='left')
    df = df.merge(df_rtf2, on=['HHID', 'LNO'], how='left')
    df = df.merge(df_rtf3, on=['HHID', 'LNO'], how='left')
    print(f'MEMBER rows: {n_member:,} | merged rows (all ages): {len(df):,}')

    df = df[(df['AGE'] >= 10) & (df['AGE'] <= 64)].reset_index(drop=True)
    print(f'Merged shape (ages 10-64): {df.shape}')
    return df

df = load_and_merge_flemms()

In [ ]:
def build_dynamic_dictionary_from_excel(v1_path, v2_path):
    # Same parser as Phase 1: {'VAR': description, 'VAR_code': 'description: value label'}
    mapping = {}
    for path, sheet in {v1_path: 'flemms_puf_2024_v1_valueset', v2_path: 'flemms_puf_2024_v2_valueset'}.items():
        try:
            dv = pd.read_excel(path, sheet_name=sheet, header=None, skiprows=1)
        except ValueError:
            dv = pd.read_excel(path, sheet_name=1, header=None, skiprows=1)
        var, desc = None, None
        for _, row in dv.iterrows():
            r = row.dropna().astype(str).str.strip().tolist()
            if not r:
                continue
            if r[0].endswith('_VS1'):
                var = r[0].replace('_VS1', '')
                desc = r[1] if len(r) > 1 else var
                mapping[var] = desc
            elif var and len(r) >= 2:
                label = r[0].split('-')[-1].strip() if '-' in r[0] else r[0]
                mapping[f"{var}_{r[-1].split('.')[0]}"] = label
    return mapping

DICT = build_dynamic_dictionary_from_excel(V1_DICT_FILE, V2_DICT_FILE)

def vlabel(col, code):
    # Value label for a coded category; falls back to the raw code
    if pd.isna(code):
        return 'Missing'
    key = f"{col}_{int(code) if float(code).is_integer() else code}"
    return f"{code}: {DICT[key]}" if key in DICT else str(code)

def vname(col):
    return DICT.get(col, col)
print(f'{len(DICT):,} dictionary entries')

## 2. Data audit

In [ ]:
print(f'Rows: {len(df):,} | Columns: {df.shape[1]} | Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
print('Duplicate (HHID, LNO) keys:', df.duplicated(['HHID', 'LNO']).sum())
print('Households:', df['HHID'].nunique())

cov = pd.Series({
    'Has FLITERATE (Form 2 record)': df['FLITERATE'].notna().mean(),
    f'Has {W}': df[W].notna().mean(),
    'Has Form 3 record (INTERNET)': df['INTERNET'].notna().mean() if 'INTERNET' in df else np.nan,
}).rename('share of rows').to_frame()
display(cov.style.format('{:.1%}'))

nun = df.nunique()
print('\nColumns by dtype:'); print(df.dtypes.astype(str).value_counts().to_string())
print('\nConstant columns:', nun[nun <= 1].index.tolist())
print('Columns with >50 unique values (dropped from PCA):', nun[nun > 50].index.tolist())

The weight `RESP_RFACT_F2` exists only for people with a Form 2 record. Rows without it cannot enter weighted rates or the model. Check how those rows differ from the analysis rows below.

In [ ]:
df['has_f2'] = df['FLITERATE'].notna()
cmp = df.groupby('has_f2').agg(n=('AGE', 'size'), mean_age=('AGE', 'mean'),
                              share_female=('SEX', lambda s: (s == 2).mean()))
display(cmp)
fig, ax = plt.subplots(figsize=(9, 3.5))
for k, g in df.groupby('has_f2'):
    sns.kdeplot(g['AGE'], ax=ax, label=f'has Form 2 = {k}', fill=True, alpha=.3)
ax.set_title('Age: people with vs without a Form 2 literacy record'); ax.legend()
savefig('02_age_by_form2_coverage')

## 3. Missingness and skip logic

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
bins = pd.cut(miss, [-.001, 0, .05, .25, .5, .75, .99, 1.0],
              labels=['0%', '0-5%', '5-25%', '25-50%', '50-75%', '75-99%', '100%'])
print(bins.value_counts().sort_index().to_string())

top = miss.head(30)
fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(x=top.values * 100, y=top.index, ax=ax, color='#4c72b0')
ax.set_xlabel('% missing'); ax.set_title('Top 30 columns by missingness')
savefig('03_missingness_top30')

Most missingness in FLEMMS is structural (skip logic): a question is asked only of a subgroup, so NaN means "not applicable", not "unknown". The heatmap shows missingness by age band for the most-missing columns; sharp age cut-offs indicate skip logic. The modeling pipeline fills these with the constant `Not_Applicable`.

In [ ]:
AGE_BINS = [10, 15, 20, 25, 35, 45, 55, 65]
AGE_LABELS = ['10-14', '15-19', '20-24', '25-34', '35-44', '45-54', '55-64']
df['age_band'] = pd.cut(df['AGE'], AGE_BINS, right=False, labels=AGE_LABELS)

cols = [c for c in miss.index if 0.01 < miss[c] < 0.999][:25]
heat = df.groupby('age_band', observed=True)[cols].agg(lambda s: s.isna().mean()).T
fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(heat, cmap='mako_r', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'share missing'})
ax.set_title('Missingness by age band (top partially-missing columns)')
savefig('03_missingness_by_age')

## 4. Target distribution

In [ ]:
# PSA: FLITERATE 1 = Literate, 2 = Illiterate. Recode to y: 0 = Literate, 1 = Illiterate
df['y'] = df['FLITERATE'].map({1: 0, 2: 1}).astype(float)
print('FLITERATE raw values:', df['FLITERATE'].value_counts(dropna=False).to_dict())

d = df.loc[df['y'].notna() & df[W].notna()]
unw = d['y'].mean()
wtd = np.average(d['y'], weights=d[W])
print(f'Analysis rows (valid target and weight): {len(d):,}')
print(f'Illiteracy prevalence, unweighted: {unw:.1%} | weighted: {wtd:.1%}')
print(f'Sum of weights (population represented): {d[W].sum():,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pd.DataFrame({'Unweighted': [1 - unw, unw], 'Weighted': [1 - wtd, wtd]}, index=['Literate', 'Illiterate']).T.plot(
    kind='bar', stacked=True, ax=axes[0], color=['#55a868', '#c44e52'], rot=0)
axes[0].set_title('Target share'); axes[0].set_ylabel('share')
sns.histplot(d[W], bins=60, ax=axes[1], color='#4c72b0'); axes[1].set_title(f'{W} distribution')
sns.histplot(np.log10(d[W]), bins=60, ax=axes[2], color='#4c72b0'); axes[2].set_title(f'log10 {W}')
savefig('04_target_and_weights')

cv = d[W].std() / d[W].mean()
ess = d[W].sum() ** 2 / (d[W] ** 2).sum()
print(f'Weight CV: {cv:.2f} | Kish effective sample size: {ess:,.0f} of {len(d):,} (design effect from weights ~ {len(d)/ess:.2f})')

Helper functions used from here on. `wrate` returns the survey-weighted illiteracy rate per category together with the unweighted rate and `n`, so thin cells are visible.

In [ ]:
def wrate(data, by):
    d = data.loc[data['y'].notna() & data[W].notna(), [by, 'y', W]].copy()
    d[by] = d[by].astype('object').where(d[by].notna(), 'Missing')
    d['wy'] = d['y'] * d[W]
    g = d.groupby(by, observed=True).agg(n=('y', 'size'), w=(W, 'sum'), wy=('wy', 'sum'), unw_rate=('y', 'mean'))
    g['w_rate'] = g['wy'] / g['w']
    g['pop_share'] = g['w'] / g['w'].sum()
    g = g.drop(columns=['wy', 'w'])
    return g.loc[sorted(g.index, key=lambda v: (v == 'Missing', 0 if v == 'Missing' else v))]  # codes ascending, Missing last

def plot_rate(ax, col, labels=True, min_n=0, order=None):
    g = wrate(df, col)
    g = g[g['n'] >= min_n]
    g = g.loc[order] if order is not None else g
    names = [vlabel(col, i)[:38] if labels and i != 'Missing' else str(i) for i in g.index]
    ax.barh(names, g['w_rate'] * 100, color='#c44e52')
    ax.scatter(g['unw_rate'] * 100, names, color='k', s=14, zorder=3, label='unweighted')
    for i, (r, n) in enumerate(zip(g['w_rate'] * 100, g['n'])):
        ax.text(r, i, f' n={n:,}', va='center', fontsize=7)
    ax.set_title(f'{col}: {vname(col)[:45]}', fontsize=9)
    ax.set_xlabel('weighted illiteracy rate (%)'); ax.invert_yaxis()

## 5. Univariate profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df['AGE'], bins=55, ax=axes[0], color='#4c72b0'); axes[0].set_title('AGE (10-64)')
for ax, col in zip(axes[1:], ['SEX', 'URBANITY']):
    vc = df[col].value_counts(normalize=True).sort_index()
    ax.bar([vlabel(col, i)[:25] for i in vc.index], vc.values)
    ax.set_title(f'{col}: {vname(col)}', fontsize=9)
savefig('05_age_sex_urbanity')

In [ ]:
# Cardinality profile: how many predictors are binary, low or high cardinality
card = df.drop(columns=['has_f2', 'age_band', 'y']).nunique()
print(pd.cut(card, [0, 2, 5, 10, 50, 1e9], labels=['<=2', '3-5', '6-10', '11-50', '>50']).value_counts().sort_index().to_string())

## 6. Bivariate: features vs target

Weighted illiteracy rate by category for a curated set of demographic, schooling, household asset, and digital variables. Dots are unweighted rates; large gaps between bar and dot flag where the design weights change the picture. Columns absent from the merged frame are skipped.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
g = wrate(df, 'age_band')
ax.bar(g.index.astype(str), g['w_rate'] * 100, color='#c44e52')
ax.plot(g.index.astype(str), g['unw_rate'] * 100, 'k.-', label='unweighted'); ax.legend()
ax.set_ylabel('weighted illiteracy rate (%)'); ax.set_title('Illiteracy by age band')
savefig('06_rate_by_age_band')
display(g.style.format({'n': '{:,}', 'unw_rate': '{:.1%}', 'w_rate': '{:.1%}', 'pop_share': '{:.1%}'}))

In [ ]:
CANDIDATES = ['SEX', 'URBANITY', 'HGC_LEVEL', 'ATTEND_SCHOOL', 'WORK', 'INTERNET', 'TVOCTRAIN', 'DAYCARE',
              'HH_PC', 'HH_REF', 'HH_WASHING', 'ROOF', 'TOILET', 'MM_WORK', 'MSTAT', 'ETHNICITY',
              'DIFF_SEEING', 'DIFF_REMEMBERING', 'DIFF_COMMUNICATING']
cand = [c for c in CANDIDATES if c in df.columns and df[c].nunique() <= 20]
print('Plotting:', cand)
ncol = 2
nrow = int(np.ceil(len(cand) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.2 * nrow))
for ax, col in zip(axes.ravel(), cand):
    plot_rate(ax, col, min_n=100)
for ax in axes.ravel()[len(cand):]:
    ax.axis('off')
savefig('06_rate_by_key_features')

Two-way view for interaction candidates (links to the Friedman H-statistic in the methodology): weighted rate by age band against schooling and internet use. Cells with `n < 50` are masked.

In [ ]:
def wrate_2d(by_row, by_col, min_n=50):
    d = df.loc[df['y'].notna() & df[W].notna(), [by_row, by_col, 'y', W]].copy()
    d['wy'] = d['y'] * d[W]
    g = d.groupby([by_row, by_col], observed=True).agg(n=('y', 'size'), w=(W, 'sum'), wy=('wy', 'sum'))
    g['rate'] = (g['wy'] / g['w']).where(g['n'] >= min_n)
    return g['rate'].unstack(by_col)

pairs = [(c, 'age_band') for c in ['HGC_LEVEL', 'INTERNET', 'ATTEND_SCHOOL'] if c in df.columns]
fig, axes = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5), squeeze=False)
for ax, (c, a) in zip(axes[0], pairs):
    t = wrate_2d(c, a) * 100
    t.index = [vlabel(c, i)[:30] for i in t.index]
    sns.heatmap(t, annot=True, fmt='.0f', cmap='rocket_r', ax=ax, cbar_kws={'label': '% illiterate'})
    ax.set_title(f'{c} x age band'); ax.set_ylabel('')
savefig('06_two_way_age')

## 7. Association ranking (Cramer's V)

Cramer's V of every predictor against the target, computed on the raw coded columns (NaN kept as its own category). Reported unweighted and design-weighted (weights rescaled to sum to n so chi-square is not inflated). This is a univariate relevance screen; it is not the model's feature selection.

In [ ]:
def cramers_v_vs_y(s, y, w=None):
    # s: categorical column, y: 0/1 array (no NaN). Returns Cramer's V (2 x k table => V = sqrt(chi2 / n))
    codes, _ = pd.factorize(s, use_na_sentinel=False)
    k = codes.max() + 1
    if k < 2:
        return np.nan
    wt = None if w is None else w * (len(y) / w.sum())
    table = np.bincount(codes * 2 + y, weights=wt, minlength=k * 2).reshape(k, 2)
    table = table[table.sum(1) > 0]
    if table.shape[0] < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    return float(np.sqrt(chi2 / table.sum()))

valid = df['y'].notna() & df[W].notna()
dv = df.loc[valid]
yv = dv['y'].astype(int).values
wv = dv[W].values

EXCLUDE = {'HHID', 'LNO', 'y', 'FLITERATE', W, 'has_f2', 'age_band'}
# Form 2 outputs are the literacy test itself (target leakage); they are not predictors
LEAKAGE = {'READING', 'WRITING', 'COMPUTE', 'COMPRE', 'READING59', 'WRITING59', 'COMPUTE59', 'LLEVEL59',
           'LLEVEL10OVER', 'LLEVEL5OVER', 'FLLEVEL', 'FLLEVEL_0', 'BASIC_NUM', 'BASIC_LIT', 'BLITERATE',
           'BLITERATE_1', 'BLITERATE_2', 'READ_IND_F2', 'RESULT_CODE_F2', 'RESP_AGREE_F2'}
ADMIN = {'REG', 'REG2', 'PRV', 'MUN', 'BGY', 'PSU', 'PSU_MEM', 'PSU_F1', 'PSU_F2', 'PSU_F3', 'MEM_RFACT',
         'HH_RFACT', 'RESP_RFACT_F3', 'RESULT_CODE_F3', 'RESP_AGREE_F3', 'RFACT', 'INTVW'}
skip = EXCLUDE | LEAKAGE | ADMIN | {c for c in df.columns if c.endswith(('_F2', '_F3'))}

rows = []
for c in df.columns:
    if c in skip:
        continue
    s = dv[c]
    if c == 'AGE':
        s = dv['age_band']
    if s.nunique(dropna=False) > 50:
        continue
    rows.append((c, s.nunique(dropna=False), cramers_v_vs_y(s, yv), cramers_v_vs_y(s, yv, wv)))
V = pd.DataFrame(rows, columns=['Feature', 'n_levels', 'V_unweighted', 'V_weighted']).sort_values('V_weighted', ascending=False)
V['Description'] = V['Feature'].map(vname)
display(V.head(30).style.format({'V_unweighted': '{:.3f}', 'V_weighted': '{:.3f}'}))

In [ ]:
top = V.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(top['Feature'], top['V_weighted'], color='#4c72b0', label='weighted')
ax.scatter(top['V_unweighted'], top['Feature'], color='k', s=14, zorder=3, label='unweighted')
ax.set_xlabel("Cramer's V vs illiteracy"); ax.set_title('Top 30 predictors by association with target'); ax.legend()
savefig('07_cramers_v_top30')

## 8. Geography

Weighted illiteracy rate by region and province. Provincial estimates with small samples are noisy, so provinces with fewer than 200 analysis rows are dropped from the ranking. Labels are PSA numeric codes; name mapping is not in the dictionary files used here `[DATA MISSING]`.

In [ ]:
g_reg = wrate(df, 'REG').sort_values('w_rate')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(g_reg.index.astype(str), g_reg['w_rate'] * 100, color='#c44e52')
ax.scatter(g_reg['unw_rate'] * 100, g_reg.index.astype(str), color='k', s=14, zorder=3)
for i, n in enumerate(g_reg['n']):
    ax.text(g_reg['w_rate'].iloc[i] * 100, i, f' n={n:,}', va='center', fontsize=7)
ax.set_xlabel('weighted illiteracy rate (%)'); ax.set_ylabel('REG code'); ax.set_title('Illiteracy by region')
savefig('08_rate_by_region')

g_prv = wrate(df, 'PRV')
print('Provinces/HUCs:', len(g_prv), '| with n < 200:', (g_prv['n'] < 200).sum())
g_prv.reset_index().to_csv('eda_province_rates.csv', index=False)

In [ ]:
gp = g_prv[g_prv['n'] >= 200].sort_values('w_rate')
sel = pd.concat([gp.head(15), gp.tail(15)])
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(sel.index.astype(str), sel['w_rate'] * 100, color=['#55a868'] * 15 + ['#c44e52'] * 15)
ax.set_xlabel('weighted illiteracy rate (%)'); ax.set_ylabel('PRV code')
ax.set_title('Lowest and highest 15 provinces/HUCs (n >= 200)')
savefig('08_rate_by_province_extremes')

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(g_prv['w_rate'] * 100, bins=25, ax=ax, color='#4c72b0')
ax.set_xlabel('weighted illiteracy rate (%)'); ax.set_title('Distribution of provincial rates (spread motivates LISA)')
savefig('08_province_rate_distribution')

## 9. Feature-feature association

Cramer's V between the top 25 predictors from section 7. Dense blocks of high V show redundancy (for example the digital skill and household asset families), which is what PCA in section 10 summarizes.

In [ ]:
def cramers_v_pair(a, b):
    ca, _ = pd.factorize(a, use_na_sentinel=False)
    cb, _ = pd.factorize(b, use_na_sentinel=False)
    ka, kb = ca.max() + 1, cb.max() + 1
    if ka < 2 or kb < 2:
        return np.nan
    t = np.bincount(ca * kb + cb, minlength=ka * kb).reshape(ka, kb)
    t = t[t.sum(1) > 0][:, t.sum(0) > 0]
    if min(t.shape) < 2:
        return np.nan
    chi2 = chi2_contingency(t, correction=False)[0]
    return float(np.sqrt(chi2 / (t.sum() * (min(t.shape) - 1))))

feats = V['Feature'].head(25).tolist()
sub = dv.sample(min(150_000, len(dv)), random_state=42)
src = {f: (sub['age_band'] if f == 'AGE' else sub[f]) for f in feats}
M = pd.DataFrame(np.nan, index=feats, columns=feats)
for i, a in enumerate(feats):
    for b in feats[i:]:
        M.loc[a, b] = M.loc[b, a] = 1.0 if a == b else cramers_v_pair(src[a], src[b])

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(M.astype(float), cmap='viridis', vmin=0, vmax=1, ax=ax, square=True, cbar_kws={'label': "Cramer's V"})
ax.set_title("Pairwise association among top 25 predictors (150k-row sample)")
savefig('09_feature_association')

## 10. PCA diagnostics

PCA is a diagnostic here, not a selector (Option C). This section one-hot encodes the predictors with the same rules as Phase 1 (drop identifiers, weights, and Form 2 leakage; categorical coding; drop columns with more than 50 levels; median and `Not_Applicable` imputation; standard scaling), fits PCA to 95% variance, and relates the components to the target.

Set `PCA_SAMPLE` to an int (for example 150000) to fit on a stratified-free random subsample if RAM or time is tight. `None` fits on all rows and should reproduce the 539 components of the main notebook.

In [ ]:
PCA_SAMPLE = None

def preprocess_for_pca_memory_safe(data, target_col='FLITERATE'):
    identifiers = [
        'HHID', 'LNO', 'REG', 'PRV', 'MUN', 'BGY', 'URBANITY', 'REG2',
        'PSU', 'PSU_MEM', 'PSU_F1', 'PSU_F2', 'PSU_F3',
        'MEM_RFACT', 'HH_RFACT', 'RESP_RFACT_F2', 'RESP_RFACT_F3',
        'RESULT_CODE_F2', 'RESP_AGREE_F2', 'READ_IND_F2',
        'READING', 'WRITING', 'COMPUTE', 'COMPRE',
        'READING59', 'WRITING59', 'COMPUTE59',
        'LLEVEL59', 'LLEVEL10OVER', 'LLEVEL5OVER', 'FLLEVEL', 'FLLEVEL_0',
        'BASIC_NUM', 'BASIC_LIT', 'BLITERATE', 'BLITERATE_1', 'BLITERATE_2',
        'RESULT_CODE_F3', 'RESP_AGREE_F3',
        'AGE_F2', 'AGE_F3', 'AGE_rtf2', 'AGE_rtf3', 'SEX_F2', 'SEX_F3',
        'REL_F2', 'REL_F3', 'HGC_GRADE_F2', 'HGC_GRADE_F3',
        # EDA helper columns added in this notebook
        'has_f2', 'age_band', 'y',
    ]
    drop_cols = [c for c in identifiers + [target_col] if c in data.columns]
    features = data.drop(columns=drop_cols)
    for col in features.columns:
        if col != 'AGE':
            s = features[col]
            try:  # integer codes as '1' not '1.0' so dummy names match the main notebook
                s = s.astype('Int64').astype('object').where(s.notna(), np.nan)
            except (TypeError, ValueError):
                s = s.astype('object')
            features[col] = s
    categorical_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = features.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

    safe_cat = []
    for col in categorical_cols:
        if features[col].nunique() > 50:
            print(f"  -> Dropping '{col}' ({features[col].nunique()} unique values)")
            features = features.drop(columns=[col])
        else:
            safe_cat.append(col)

    if numeric_cols:
        features[numeric_cols] = SimpleImputer(strategy='median').fit_transform(features[numeric_cols])
    if safe_cat:
        features[safe_cat] = SimpleImputer(strategy='constant', fill_value='Not_Applicable').fit_transform(features[safe_cat])

    enc = pd.get_dummies(features, columns=safe_cat, drop_first=True, dtype=np.uint8)
    del features
    gc.collect()
    names = enc.columns
    X = StandardScaler().fit_transform(enc).astype(np.float32)
    del enc
    gc.collect()
    print(f'Encoded + scaled: {X.shape}')
    return X, names

X_scaled, feature_names = preprocess_for_pca_memory_safe(df)
y_all = df['y'].values
w_all = df[W].values

if PCA_SAMPLE and PCA_SAMPLE < len(X_scaled):
    fit_idx = RNG.choice(len(X_scaled), PCA_SAMPLE, replace=False)
else:
    fit_idx = np.arange(len(X_scaled))

pca = PCA(n_components=0.95, random_state=42).fit(X_scaled[fit_idx])
print(f'Components for 95% variance: {pca.n_components_} (fit rows: {len(fit_idx):,})')

In [ ]:
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].bar(range(1, 31), evr[:30] * 100, color='#4c72b0'); axes[0].set_title('Scree (first 30 components)')
axes[0].set_xlabel('component'); axes[0].set_ylabel('% variance')
axes[1].plot(range(1, len(cum) + 1), cum, color='#4c72b0')
for t in (0.5, 0.8, 0.9, 0.95):
    k = int(np.searchsorted(cum, t) + 1)
    axes[1].axhline(t, color='grey', ls=':'); axes[1].axvline(k, color='grey', ls=':')
    axes[1].text(k, t - .06, f'{int(t*100)}%: {k}', fontsize=8)
axes[1].set_title('Cumulative explained variance'); axes[1].set_xlabel('components')
axes[2].plot(range(1, 101), np.cumsum(evr[:100]) * 100, color='#4c72b0')
axes[2].set_title('Cumulative variance, first 100 components'); axes[2].set_xlabel('components')
savefig('10_pca_variance')
print(f'PC1 explains {evr[0]:.1%}, top 10 explain {cum[9]:.1%}')

Loadings of the first 10 components. Each panel shows the 8 largest absolute loadings (the main notebook takes the top 5 per component for its PCA diagnostic list).

In [ ]:
def short(n):
    base = n.rsplit('_', 1)[0] if re.search(r'_\s?-?\d+$', n) else n
    code_ = n[len(base) + 1:].strip() if base != n else ''
    lab = DICT.get(base, base)[:28]
    return f'{n} ({lab})' if code_ else n

fig, axes = plt.subplots(5, 2, figsize=(15, 17))
for i, ax in enumerate(axes.ravel()):
    load = pca.components_[i]
    idx = np.argsort(np.abs(load))[::-1][:8]
    ax.barh([short(feature_names[j]) for j in idx][::-1], load[idx][::-1],
            color=['#c44e52' if v > 0 else '#4c72b0' for v in load[idx][::-1]])
    ax.set_title(f'PC{i+1} ({evr[i]:.1%} variance)', fontsize=10); ax.tick_params(labelsize=7)
savefig('10_pca_loadings_top10')

Variance versus relevance: absolute weighted correlation of each component score with `y`, next to its explained variance. If the target-relevant components are not the high-variance ones, PCA ranking is a poor selector, which is the panel's "variance is not relevance" point. Scores for rows with a valid target and weight only.

In [ ]:
mask = ~np.isnan(y_all) & ~np.isnan(w_all)
K = 40
Z = pca.transform(X_scaled[mask])[:, :K]
ym, wm = y_all[mask], w_all[mask]

def wcorr(a, b, w):
    ma, mb = np.average(a, weights=w), np.average(b, weights=w)
    cov = np.average((a - ma) * (b - mb), weights=w)
    return cov / np.sqrt(np.average((a - ma) ** 2, weights=w) * np.average((b - mb) ** 2, weights=w))

corr = np.array([wcorr(Z[:, k], ym, wm) for k in range(K)])
auc = np.array([roc_auc_score(ym, Z[:, k], sample_weight=wm) for k in range(K)])
auc = np.maximum(auc, 1 - auc)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].bar(range(1, K + 1), evr[:K] * 100, color='#4c72b0'); axes[0].set_title('Explained variance (%)')
axes[1].bar(range(1, K + 1), np.abs(corr), color='#c44e52'); axes[1].set_title('|weighted correlation| with illiteracy')
axes[2].scatter(evr[:K] * 100, auc, c=range(K), cmap='viridis')
for k in np.argsort(-auc)[:6]:
    axes[2].annotate(f'PC{k+1}', (evr[k] * 100, auc[k]))
axes[2].set_xscale('log'); axes[2].set_xlabel('% variance (log)'); axes[2].set_ylabel('single-component weighted AUC')
axes[2].set_title('Variance vs single-component AUC')
for ax in axes[:2]:
    ax.set_xlabel('component')
savefig('10_pca_variance_vs_relevance')
print('Top components by AUC:', [(f'PC{k+1}', round(float(auc[k]), 3)) for k in np.argsort(-auc)[:5]])
print('Rank correlation (variance order vs |corr|):',
      round(float(pd.Series(np.abs(corr)).corr(pd.Series(-np.arange(K)), method='spearman')), 3))

In [ ]:
# PC1 vs PC2, colored by target (random 20k rows with valid target)
sidx = RNG.choice(len(Z), min(20_000, len(Z)), replace=False)
Zs = pca.transform(X_scaled[mask][sidx])[:, :2]
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Zs[:, 0], Zs[:, 1], c=np.where(ym[sidx] == 1, '#c44e52', '#55a868'), s=3, alpha=.35)
ax.set_xlabel(f'PC1 ({evr[0]:.1%})'); ax.set_ylabel(f'PC2 ({evr[1]:.1%})')
ax.set_title('PC1 vs PC2 (red = illiterate, green = literate; unweighted sample)')
savefig('10_pc1_pc2_scatter')
del Z, Zs
gc.collect()

Overlap between the PCA-loading feature list (Phase 1) and the supervised gain list (Phase 1B), read from the checked-in CSVs.

In [ ]:
f_pca = pd.read_csv(PHASE1_FILE)
f_sup = pd.read_csv(PHASE1B_FILE)
a = set(f_pca['Feature_Name'].str.strip()); b = set(f_sup['Feature_Name'].str.strip())
both = sorted(a & b)
print(f'PCA list: {len(a)} | supervised list: {len(b)} | overlap: {len(both)} ({len(both)/len(b):.0%} of supervised)')
display(pd.DataFrame({'Feature_Name': both}).merge(f_sup, on='Feature_Name'))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['PCA only', 'Both', 'Supervised only'], [len(a - b), len(both), len(b - a)], color=['#4c72b0', '#8172b2', '#c44e52'])
ax.set_title('Feature list overlap'); ax.set_ylabel('features')
savefig('10_pca_vs_supervised_overlap')

## Outputs

- Figures: `eda_figures/*.png` (200 dpi)
- Table: `eda_province_rates.csv` (weighted and unweighted illiteracy rate and n per `PRV` code, input for the LISA step)